# 08 — LLM-powered features

memtomem can use an LLM to enhance three features:

1. **Auto-tagging** — semantic tag extraction (vs keyword frequency)
2. **Entity extraction** — structured extraction of people, dates, decisions
3. **Query expansion** — LLM-generated synonyms before search

All three are optional and gracefully fall back to heuristics when LLM
is disabled or fails.  This notebook demonstrates both paths.

**Requirements:** Ollama running with a chat model (e.g. `llama3.2`).
If Ollama is not available, the notebook still runs — it shows the
heuristic fallback for each feature.

## Prerequisites

```bash
uv pip install "memtomem[ollama]" jupyter ipykernel
ollama serve
ollama pull nomic-embed-text  # embeddings
ollama pull llama3.2          # LLM (chat)
```

In [ ]:
import httpx

_ollama_ok = False
try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=2)
    r.raise_for_status()
    _ollama_ok = True
    print("Ollama is up — LLM features will use llama3.2.")
except Exception:
    print("Ollama not reachable — LLM features will show heuristic fallback only.")

## Setup: components + corpus

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import memtomem.config as _cfg
from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import Components, create_components, close_components

tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

corpus = {
    "meeting_2026_04.md": (
        "# Team meeting — April 2026\n\n"
        "Attendees: Alice Kim, Bob Chen, Carol Park\n\n"
        "Decision: We will migrate the auth service to OAuth2 by 2026-06-01.\n\n"
        "Action item: Bob to draft the migration plan by next Friday.\n\n"
        "TODO: Carol reviews the OpenID Connect spec for Korean locale support.\n"
    ),
    "architecture.md": (
        "# System architecture\n\n"
        "The backend uses FastAPI with PostgreSQL. The frontend is a Next.js\n"
        "application deployed on Vercel. Redis handles session caching and\n"
        "rate limiting. Kubernetes orchestrates the backend services.\n"
    ),
    "incident.md": (
        "# Incident 2026-03-15\n\n"
        "The deployment on March 15th caused a p99 latency spike from 200ms to 3s.\n"
        "Root cause: the database migration locked the users table for 45 seconds.\n\n"
        "Resolved: We rolled back and re-applied the migration with online DDL.\n"
    ),
}

for name, content in corpus.items():
    (notes_dir / name).write_text(content, encoding="utf-8")

config = Mem2MemConfig()
config.storage.sqlite_path = db_path
config.indexing.memory_dirs = [notes_dir]

# Use BM25-only embeddings (works without Ollama embedding model)
config.embedding.provider = "none"
config.embedding.model = ""
config.embedding.dimension = 0

# Enable LLM if Ollama is available
if _ollama_ok:
    config.llm.enabled = True
    config.llm.provider = "ollama"
    config.llm.model = "llama3.2"

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None

comp: Components = await create_components(config)
await comp.index_engine.index_path(notes_dir)

llm_status = "enabled (llama3.2)" if comp.llm else "disabled (heuristic fallback)"
print(f"Components ready. LLM: {llm_status}")

## Feature 1: Auto-tagging

With LLM enabled, `mem_auto_tag` generates semantically meaningful tags.
Without LLM, it falls back to keyword frequency analysis.

In [ ]:
from memtomem.tools.auto_tag import extract_tags_keyword, _extract_tags_with_fallback

sample = corpus["meeting_2026_04.md"]

# Heuristic (always available)
kw_tags = extract_tags_keyword(sample, max_tags=5)
print(f"Keyword tags: {kw_tags}")

# With LLM fallback — uses LLM if available, keyword otherwise
smart_tags = await _extract_tags_with_fallback(sample, max_tags=5, llm_provider=comp.llm)
print(f"Smart tags:   {smart_tags}")

## Feature 2: Entity extraction

Extract structured entities — people, dates, decisions, action items,
technologies.  LLM extraction is more accurate for names and decisions;
regex handles dates and known tech keywords well.

In [ ]:
from memtomem.tools.entity_extraction import (
    extract_entities,
    extract_entities_with_llm,
)

sample = corpus["meeting_2026_04.md"]

# Heuristic (regex patterns)
heuristic = extract_entities(sample)
print("--- Heuristic entities ---")
for e in heuristic:
    print(f"  {e.entity_type:12s} {e.entity_value:40s} conf={e.confidence:.2f}")

print()

# With LLM (if available)
smart = await extract_entities_with_llm(sample, llm_provider=comp.llm)
print("--- Smart entities (LLM + fallback) ---")
for e in smart:
    print(f"  {e.entity_type:12s} {e.entity_value:40s} conf={e.confidence:.2f}")

## Feature 3: Query expansion

LLM query expansion adds synonyms and related terms to the search query
before retrieval.  This helps when the user's query uses different words
than the indexed documents.

The expansion function has a **3-second hard timeout** to prevent search
latency blow-up.

In [ ]:
from memtomem.search.expansion import expand_query_llm, expand_query_tags

query = "authentication migration deadline"

# Tag-based expansion (heuristic, always available)
tag_expanded = await expand_query_tags(query, comp.storage, max_terms=3)
print(f"Tag expansion:  {tag_expanded!r}")

# LLM expansion (if available)
if comp.llm:
    try:
        llm_expanded = await expand_query_llm(query, comp.llm, max_terms=3)
        print(f"LLM expansion:  {llm_expanded!r}")
    except Exception as e:
        print(f"LLM expansion failed: {e}")
else:
    print("LLM not available — skipping LLM expansion demo.")

## Configuring LLM providers

The `LLMConfig` section supports three providers:

```python
# Ollama (local, default)
config.llm.enabled = True
config.llm.provider = "ollama"
config.llm.model = "llama3.2"

# OpenAI (cloud) — also works with LM Studio, vLLM, OpenRouter
config.llm.provider = "openai"
config.llm.model = "gpt-4o-mini"
config.llm.api_key = "sk-..."
config.llm.base_url = "https://api.openai.com"  # or http://localhost:1234 for LM Studio

# Anthropic (cloud)
config.llm.provider = "anthropic"
config.llm.model = "claude-sonnet-4-20250514"
config.llm.api_key = "sk-ant-..."
```

See [LLM Providers guide](../../docs/guides/llm-providers.md) for
environment variable configuration and OpenAI-compatible endpoint setup.

## Cleanup

In [ ]:
await close_components(comp)
_cfg.load_config_overrides = _orig_loader
tmp.cleanup()
print("clean.")